# M18K V2 | Dataset integrity and exploratory analysis

**Research record · 30 October 2026**  
**Reference:** Zakeri *et al.* (2025), “M18K: A Multi-Purpose Real-World Dataset for Mushroom Detection, 3D Pose Estimation, and Growth Monitoring,” *Computers* 14(5), 199. [DOI](https://doi.org/10.3390/computers14050199) · [Source repository](https://github.com/abdollahzakeri/m18k)

### Research question
What are the composition, annotation quality, RGB–depth coverage, and evaluation risks of the expanded M18K V2 release, and is it suitable for reproducible instance-segmentation experiments?

### Scope and evidence
This notebook audits the **V2 release**, not the exact 2,000-image subset used in the paper. The original benchmark used a 70/10/20 train/validation/unseen-test protocol; the downloaded V2 release supplies its own splits. Segmentation AP figures in Table 2 (YOLOv8-N: **0.712 at 640** and **0.846 at 1280**) are reference values, **not reproduced scores**.

This notebook is divided into **dataset discovery → one-pass annotation QC → EDA → evidence export → optional model experiment**. The model stages are disabled by default. No test-set performance is claimed until a trained checkpoint is evaluated on the held-out test split.

## 1. Reproducibility settings

Dataset and EDA outputs are stored on Google Drive. Model preparation and training are intentionally off today; enable them for a later experiment. The input path matches the extracted archive structure verified in Colab.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json, csv, math, random, hashlib, shutil, os, sys, time, platform, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DRIVE_ROOT = Path('/content/drive/MyDrive/M18K_Reproduction')
DATA_ROOT = DRIVE_ROOT / 'M18K_V2_data' / 'M18KV2'
REPORT_DIR = DRIVE_ROOT / 'reports' / 'v2_eda'
WORK_ROOT = Path('/content/M18K_V2_work') if Path('/content').exists() else Path.cwd() / 'M18K_V2_work'

# Today's run: data audit and EDA only. These remain false until training day.
PREPARE_YOLO = True
RUN_TRAIN = True
RUN_TEST_EVAL = True
RUN_FULL_CONTENT_HASH_AUDIT = False  # slow on mounted Drive; optional separate audit

# Model protocol for a future pilot (not paper-identical).
MODEL_NAME = 'yolov8n-seg.pt'
IMAGE_SIZE, EPOCHS, BATCH = 640, 10, 8

WORK_ROOT.mkdir(parents=True, exist_ok=True)
print('Python:', platform.python_version())
print('Training:', RUN_TRAIN, '| YOLO export:', PREPARE_YOLO)

## 2. Data provenance and file inventory

This analysis checks the existing `train/`, `valid/`, and `test/` directories, where images are stored in `rgb/` and depth maps in `depth/`.

In [ ]:
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except ImportError:
    pass  # Allow inspection outside Colab when DATA_ROOT is changed.

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f'V2 extraction not found: {DATA_ROOT}')

REPORT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIRS = {'train': 'train', 'valid': 'val', 'test': 'test'}
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}
DEPTH_EXTS = {'.npy', '.npz', '.png', '.tif', '.tiff', '.exr'}

# A shallow directory listing is much faster on Drive than repeated recursive lookups.
def catalog(folder, extensions):
    if not folder.is_dir(): return []
    return sorted((p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in extensions), key=lambda p: p.name)

file_index, COCO_FILES = {}, {}
inventory = []
for folder_name, split in SPLIT_DIRS.items():
    base = DATA_ROOT / folder_name
    jp = base / 'annotations_coco.json'
    if not jp.is_file():
        candidates = sorted(base.glob('*coco*.json'))
        if not candidates:
            raise FileNotFoundError(f'COCO JSON not found for {split}: {base}')
        jp = candidates[0]
    rgb_dir, depth_dir = base/'rgb', base/'depth'
    rgb = catalog(rgb_dir, IMAGE_EXTS)
    depth = catalog(depth_dir, DEPTH_EXTS)
    if not rgb:
        raise FileNotFoundError(f'RGB images not found in {rgb_dir}')

    def key_by_stem(paths):
        out = defaultdict(list)
        for p in paths: out[p.stem.casefold()].append(p)
        return out

    file_index[split] = {
        'rgb': rgb, 'depth': depth,
        'rgb_by_stem': key_by_stem(rgb),
        'depth_by_stem': key_by_stem(depth),
        'rgb_by_name': {p.name.casefold():p for p in rgb},
    }
    COCO_FILES[split] = jp
    inventory.append({'split':split,'rgb_files':len(rgb),'depth_files':len(depth),
                      'json_GiB':round(jp.stat().st_size/2**30,3),
                      'annotation_file':str(jp)})

inventory_df = pd.DataFrame(inventory)
display(inventory_df[['split','rgb_files','depth_files','json_GiB']])
print('Indexed file paths, not image pixels. No archive download needed.')

## 3. Streaming COCO inspection

The core annotation audit reads each large COCO JSON **once**, item by item. A separate pass is used later to retrieve masks for selected visual examples.

In [ ]:
import subprocess
try:
    import ijson
    from ijson.common import ObjectBuilder
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'ijson'])
    import ijson
    from ijson.common import ObjectBuilder

# Parse three relevant COCO arrays in a single sequential pass.
def coco_records(json_path):
    targets = {'images.item', 'annotations.item', 'categories.item'}
    active = None
    builder = None
    with Path(json_path).open('rb') as fh:
        for prefix, event, value in ijson.parse(fh, use_float=True):
            if active is None:
                if prefix in targets and event == 'start_map':
                    active = prefix
                    builder = ObjectBuilder()
                    builder.event(event, value)
            else:
                builder.event(event, value)
                if prefix == active and event == 'end_map':
                    yield active.split('.')[0], builder.value
                    active, builder = None, None

CLASS_ALIASES = {
    'bb':'BB', 'baby bella':'BB', 'baby-bella':'BB', 'brown button':'BB', 'cremini':'BB',
    'wb':'WB', 'white button':'WB', 'white mushroom':'WB'
}
def canonical_class(raw_name):
    return CLASS_ALIASES.get(str(raw_name).strip().lower().replace('_',' '))

def resolve_rgb(info, split):
    raw = str(info.get('file_name','')).replace('\\','/')
    base = Path(raw).name.casefold()
    match = file_index[split]['rgb_by_name'].get(base)
    if match is not None:
        return match, 'exact_name'
    candidates = file_index[split]['rgb_by_stem'].get(Path(raw).stem.casefold(), [])
    return (candidates[0], 'stem_only') if len(candidates)==1 else (None, 'unresolved')

def resolve_depth(rgb_path, split):
    if rgb_path is None: return None, 'rgb_missing'
    candidates = file_index[split]['depth_by_stem'].get(rgb_path.stem.casefold(), [])
    if len(candidates) == 1:
        return candidates[0], 'matched_by_stem'
    return None, ('no_depth_match' if not candidates else 'ambiguous_depth_match')

def bbox_problem(box, info):
    if not isinstance(box, list) or len(box)!=4:
        return 'bad_bbox_format'
    try:
        x,y,w,h = map(float, box)
    except (TypeError, ValueError):
        return 'bad_bbox_values'
    if not all(math.isfinite(v) for v in (x,y,w,h)):
        return 'nonfinite_bbox'
    if w<=0 or h<=0: return 'nonpositive_bbox'
    W,H = float(info.get('width',0)), float(info.get('height',0))
    if W<=0 or H<=0: return 'unknown_image_dimensions'
    if x<0 or y<0 or x+w>W+2 or y+h>H+2: return 'bbox_outside_image'
    return None

def seg_problem(seg):
    if seg is None or seg==[] or seg=={}: return 'missing_segmentation'
    if isinstance(seg,dict):
        return None if ('counts' in seg and 'size' in seg) else 'bad_RLE'
    if not isinstance(seg,list): return 'unknown_segmentation_type'
    if not seg: return 'missing_segmentation'
    for part in seg:
        if not isinstance(part,list) or len(part)<6 or len(part)%2:
            return 'bad_polygon_shape'
    return None

# Reservoir sampling yields reproducible plots without keeping all 700k boxes.
class Reservoir:
    def __init__(self, max_items=12000, seed=42):
        self.max_items, self.n, self.items = max_items, 0, []
        self.rng = random.Random(seed)
    def add(self, value):
        self.n += 1
        if len(self.items)<self.max_items:
            self.items.append(value)
        else:
            j = self.rng.randrange(self.n)
            if j < self.max_items: self.items[j] = value

image_rows = []
images_by_key = {}
raw_category_names = {}     # (split, raw_id) -> source name
raw_class_counts = Counter()
quality = Counter()
segmentation_types = Counter()
quality_examples = []
bbox_samples = defaultdict(lambda: Reservoir(12000, SEED))
annotation_counts_by_split = Counter()

for folder_name, split in SPLIT_DIRS.items():
    coco_path = COCO_FILES[split]
    image_info = {}
    annotation_index = 0
    source_categories = {}
    print(f'\nReading {split}: {coco_path.name} ({coco_path.stat().st_size/2**20:.1f} MiB)')
    t0 = time.time()
    for kind, item in coco_records(coco_path):
        if kind == 'images':
            iid = int(item['id'])
            if iid in image_info:
                raise ValueError(f'Duplicate image id {iid} in {coco_path}')
            path, match_type = resolve_rgb(item, split)
            depth, depth_status = resolve_depth(path, split)
            row = {
                'split':split, 'image_id':iid, 'coco_filename':str(item.get('file_name','')),
                'image_path':str(path) if path else '',
                'depth_path':str(depth) if depth else '',
                'rgb_match':match_type, 'depth_status':depth_status,
                'width':int(item.get('width') or 0), 'height':int(item.get('height') or 0),
                'instances':0, 'raw_counts':Counter(),
            }
            image_info[iid] = row
            image_rows.append(row)
            images_by_key[(split,iid)] = row
        elif kind == 'categories':
            cid = int(item['id'])
            source_categories[cid] = str(item.get('name',''))
        elif kind == 'annotations':
            annotation_index += 1
            iid = int(item['image_id'])
            cid = int(item['category_id'])
            annotation_counts_by_split[split] += 1
            raw_class_counts[(split,cid)] += 1
            row = image_info.get(iid)
            if row is None:
                quality['annotation_references_unknown_image'] += 1
                if len(quality_examples) < 40:
                    quality_examples.append({'split':split, 'problem':'unknown_image','image_id':iid,
                                             'annotation_id':item.get('id')})
                continue
            row['instances'] += 1
            row['raw_counts'][cid] += 1
            issue = bbox_problem(item.get('bbox'), row)
            if issue:
                quality[issue] += 1
                if len(quality_examples) < 40:
                    quality_examples.append({'split':split, 'problem':issue, 'image_id':iid,
                                             'annotation_id':item.get('id')})
            else:
                box = item['bbox']
                area_fraction = float(box[2])*float(box[3]) / (row['width']*row['height'])
                if area_fraction > 0:
                    bbox_samples[(split,cid)].add(area_fraction)
            seg = item.get('segmentation')
            segmentation_types['polygon' if isinstance(seg,list) else 'RLE' if isinstance(seg,dict) else 'other'] += 1
            if isinstance(seg,list) and len(seg)>1:
                segmentation_types['multi_polygon_instances'] += 1
            seg_issue = seg_problem(seg)
            if seg_issue: quality[seg_issue] += 1
            if annotation_index % 100000 == 0:
                print(f'  {annotation_index:,} annotations processed ({time.time()-t0:.0f}s)')

    for cid, name in source_categories.items():
        raw_category_names[(split,cid)] = name
    print(f'  Done: {len(image_info):,} images; {annotation_index:,} annotations; {time.time()-t0:.1f}s')

# Ground the labels in each COCO file, never infer them from numeric IDs alone.
class_mapping = {}
for (split,cid), raw_name in sorted(raw_category_names.items()):
    class_mapping[(split,cid)] = canonical_class(raw_name)

raw_class_table = pd.DataFrame([
    {'split':s, 'raw_category_id':cid, 'category_name':raw_category_names.get((s,cid),'[absent]'),
     'mapped_class':class_mapping.get((s,cid)), 'annotations':count}
    for (s,cid),count in sorted(raw_class_counts.items())
])
print('\nOriginal category counts:')
display(raw_class_table)

unmapped = raw_class_table[raw_class_table['mapped_class'].isna()]
if not unmapped.empty:
    raise ValueError('Annotations use an unmapped class. Review raw category names before EDA or training.')

for row in image_rows:
    row['BB'] = sum(n for cid,n in row['raw_counts'].items()
                    if class_mapping.get((row['split'],cid))=='BB')
    row['WB'] = sum(n for cid,n in row['raw_counts'].items()
                    if class_mapping.get((row['split'],cid))=='WB')
    del row['raw_counts']

print('Images:',len(image_rows),'| Annotations:',sum(annotation_counts_by_split.values()))
print('QC flags:',dict(quality))
print('Segmentation representations:',dict(segmentation_types))

## 4. Split integrity and RGB–depth coverage

We retain the dataset-provided split (`train`, `valid`→`val`, `test`) without reshuffling. Pairing uses filename stems **within each corresponding split** and supports the actual `rgb/` and `depth/` folders. This verifies file pairing, not calibration, metric depth units or scene-level independence.

In [ ]:
manifest_df = pd.DataFrame(image_rows)
manifest_df.to_csv(REPORT_DIR/'image_manifest.csv', index=False)
raw_class_table.to_csv(REPORT_DIR/'raw_category_audit.csv', index=False)
pd.DataFrame(quality_examples).to_csv(REPORT_DIR/'annotation_error_examples.csv', index=False)

split_summary = manifest_df.groupby('split').agg(
    images=('image_id','size'), annotations=('instances','sum'),
    BB=('BB','sum'), WB=('WB','sum'),
    images_with_BB=('BB',lambda x:int((x>0).sum())),
    rgb_unresolved=('image_path',lambda x:int((x=='').sum())),
    depth_matched=('depth_path',lambda x:int((x!='').sum())),
).reindex(['train','val','test'])
split_summary['depth_coverage_pct'] = (split_summary['depth_matched']/split_summary['images']*100).round(2)
split_summary.to_csv(REPORT_DIR/'split_summary.csv')
display(split_summary)

# Check the files by filename in their split; SHA-256 is offered separately.
by_filename = defaultdict(set)
for row in image_rows:
    if row['image_path']:
        by_filename[Path(row['image_path']).name.casefold()].add(row['split'])
filename_overlap = {k: sorted(v) for k,v in by_filename.items() if len(v)>1}

# Inspect actual matching and non-matching filenames, without assuming they pair.
for split in ('train','val','test'):
    subset = manifest_df[manifest_df['split']==split]
    no_depth = subset[subset['depth_path']=='']
    print(f'{split}: depth matched {len(subset)-len(no_depth)}/{len(subset)};'
          f' sample unmatched RGB:', [Path(x).name for x in no_depth['image_path'].head(3)])
    if len(no_depth):
        print('  depth files available:', [p.name for p in file_index[split]['depth'][:3]])

print('Exact filename overlaps between splits:',len(filename_overlap))
if filename_overlap:
    print('Examples:', list(filename_overlap.items())[:5])
print('Full image-content hash audit completed:', RUN_FULL_CONTENT_HASH_AUDIT)

if manifest_df['image_path'].eq('').any():
    raise FileNotFoundError('Some RGB images did not resolve; see image_manifest.csv.')
if manifest_df['depth_path'].eq('').any():
    print('WARNING: Not all RGB images have a matched depth file. Pairing is incomplete.')

### Optional: exact duplicate-image audit

The earlier notebook spent substantial time hashing images through Drive. This full audit is therefore **opt-in**. Matching filenames are only a warning, not proof of matching image content. Even complete SHA-256 checks cannot exclude *near*-duplicate frames from the same growing scene.

In [ ]:
if RUN_FULL_CONTENT_HASH_AUDIT:
    from tqdm.auto import tqdm
    hashes = defaultdict(set)
    hash_failures = []
    for row in tqdm(image_rows, desc='Hashing RGB files (slow on Drive)'):
        try:
            dig = hashlib.sha256()
            with open(row['image_path'],'rb') as fh:
                for chunk in iter(lambda:fh.read(2*1024*1024),b''):
                    dig.update(chunk)
            hashes[dig.hexdigest()].add(row['split'])
        except OSError as e:
            hash_failures.append((row['image_path'],str(e)))
    duplicate_hashes = {digest:sorted(v) for digest,v in hashes.items() if len(v)>1}
    pd.DataFrame([{'sha256':k,'splits':','.join(v)} for k,v in duplicate_hashes.items()]).to_csv(
        REPORT_DIR/'cross_split_sha256_duplicates.csv',index=False)
    print('Cross-split exact-content duplicates:',len(duplicate_hashes))
    print('Unreadable files:',len(hash_failures))
else:
    duplicate_hashes = None
    print('Full SHA-256 duplicate audit was not run. Cross-split image leakage remains unverified.')

## 5. Exploratory analysis

The analysis reports all instances rather than just the easy-to-see large mushrooms. Because the sample is dense and BB appears rare, the plots use both absolute counts and a logarithmic size scale. Box-size distributions use an explicitly labeled reservoir sample; the class and image counts are exact.

In [ ]:
class_counts = manifest_df[['BB','WB']].sum().astype(int)
class_pct = (class_counts/class_counts.sum()*100).round(3)
print('Class distribution, mapped from COCO category names:')
display(pd.DataFrame({'instances':class_counts,'percent':class_pct}))
print('Images containing BB:',int((manifest_df.BB>0).sum()), '/',len(manifest_df))

fig,axes = plt.subplots(2,2,figsize=(13,9))
counts = split_summary['images']
axes[0,0].bar(counts.index, counts.values)
axes[0,0].set(title='RGB images by supplied split',ylabel='Images')
axes[0,1].bar(['BB','WB'],[int(class_counts.BB),int(class_counts.WB)])
axes[0,1].set(title='COCO instances by class (log scale)', ylabel='Instances',yscale='log')
axes[1,0].hist(manifest_df['instances'], bins=40)
axes[1,0].set(title='Mushroom instances per image',xlabel='Instances',ylabel='Images')
for klass in ('BB','WB'):
    box_values=[]
    for (split,cid),res in bbox_samples.items():
        if class_mapping.get((split,cid))==klass:
            box_values.extend(res.items)
    if box_values:
        axes[1,1].hist(box_values,bins=np.logspace(-7,0,45),alpha=.6,label=f'{klass} (sample)')
axes[1,1].set_xscale('log')
axes[1,1].set(title='Bounding-box area / image area (sample)',xlabel='Fraction',ylabel='Sampled boxes')
axes[1,1].legend()
for ax in axes.flat: ax.grid(alpha=.16,axis='y')
fig.suptitle('M18K V2 — Dataset composition',fontsize=15)
fig.tight_layout()
fig.savefig(REPORT_DIR/'eda_composition.png',dpi=160,bbox_inches='tight')
plt.show()

# Presence of depth files and image-density statistics.
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(split_summary.index,split_summary['depth_coverage_pct'])
ax.set(title='RGB-to-depth file pairing',ylabel='Matched RGB images (%)',ylim=(0,105))
fig.tight_layout()
fig.savefig(REPORT_DIR/'eda_depth_coverage.png',dpi=160,bbox_inches='tight')
plt.show()

print('Instances per image (quantiles):')
display(manifest_df['instances'].quantile([0,.10,.25,.5,.75,.90,1]).rename('instances'))

## 6. RGB, masks and depth: visual checks

Use examples from each split, prioritizing images that contain BB for rare-class verification. The depth display normalizes valid values for visualization only; **depth units are not inferred**. A readable depth file is not proof of correct sensor alignment.

In [ ]:
from matplotlib.patches import Rectangle

def load_depth(path):
    p=Path(path)
    if p.suffix.lower()=='.npy':
        arr=np.load(p,allow_pickle=False, mmap_mode='r')
    elif p.suffix.lower()=='.npz':
        with np.load(p, allow_pickle=False) as z:
            if len(z.files)!=1: raise ValueError(f'Ambiguous NPZ depth keys: {z.files}')
            arr=z[z.files[0]]
    else:
        with Image.open(p) as img:
            arr=np.asarray(img)
    arr=np.squeeze(arr)
    if arr.ndim!=2: raise ValueError(f'Depth is not a 2D plane: {p}, {arr.shape}')
    return np.asarray(arr)

sampled = []
for split in ('train','val','test'):
    subset=manifest_df[manifest_df.split==split]
    preferred=subset[subset.BB>0]
    if len(preferred): sampled.append(preferred.iloc[0])
    if len(subset): sampled.append(subset.iloc[len(subset)//2])

fig, axs=plt.subplots(len(sampled),2,figsize=(11,4*len(sampled)),squeeze=False)
depth_examples=[]
for i,row in enumerate(sampled):
    with Image.open(row['image_path']) as im:
        rgb=np.asarray(im.convert('RGB'))
    axs[i,0].imshow(rgb)
    axs[i,0].set_title(f"{row['split']} | RGB | BB={row['BB']}, WB={row['WB']}")
    axs[i,0].axis('off')
    if row['depth_path']:
        try:
            d=load_depth(row['depth_path'])
            finite = np.isfinite(d) & (d>0)
            valid=d[finite]
            vmin,vmax=np.percentile(valid,[1,99]) if len(valid) else (None,None)
            axs[i,1].imshow(d,cmap='viridis',vmin=vmin,vmax=vmax)
            axs[i,1].set_title(f"Depth | shape={d.shape} | valid={finite.mean():.1%}")
            depth_examples.append({'split':row['split'],'image_id':row['image_id'],
                                   'depth_shape':str(d.shape),'dtype':str(d.dtype),
                                   'positive_finite_fraction':float(finite.mean()),
                                   'rgb_shape':str(rgb.shape[:2]),
                                   'shape_aligned':bool(d.shape==rgb.shape[:2])})
        except Exception as e:
            axs[i,1].text(.1,.5,str(e)[:160],transform=axs[i,1].transAxes)
            axs[i,1].set_title('Depth read failed')
    else:
        axs[i,1].text(.12,.5,'Unmatched depth filename',transform=axs[i,1].transAxes)
        axs[i,1].set_title('No paired depth')
    axs[i,1].axis('off')
fig.tight_layout()
fig.savefig(REPORT_DIR/'rgb_depth_examples.png',dpi=140,bbox_inches='tight')
plt.show()
pd.DataFrame(depth_examples).to_csv(REPORT_DIR/'depth_sample_audit.csv',index=False)

### Instance-mask overlay, selected examples

Polygons are read only for the selected images, keeping RAM bounded. This plot is a qualitative geometry check, not a segmentation accuracy metric.

In [ ]:
# Re-read only selected annotation records (necessary because the first pass discarded full polygons).
selected = {(str(row['split']),int(row['image_id'])) for row in sampled}
chosen = defaultdict(list)
for split in ('train','val','test'):
    ids={iid for sp,iid in selected if sp==split}
    if not ids: continue
    for kind, ann in coco_records(COCO_FILES[split]):
        if kind=='annotations' and int(ann['image_id']) in ids:
            if len(chosen[(split,int(ann['image_id']))])<300:
                chosen[(split,int(ann['image_id']))].append(ann)

fig,axs=plt.subplots(2,3,figsize=(16,9));axs=axs.ravel()
for ax,(_,row) in zip(axs,enumerate(sampled[:6])):
    with Image.open(row['image_path']) as im:
        ax.imshow(np.asarray(im.convert('RGB')))
    for a in chosen[(row['split'],int(row['image_id']))]:
        cls=class_mapping.get((row['split'],int(a['category_id'])))
        color = 'cyan' if cls=='BB' else 'orange'
        seg=a.get('segmentation')
        if isinstance(seg,list):
            for poly in seg:
                if isinstance(poly,list) and len(poly)>=6 and len(poly)%2==0:
                    pts=np.asarray(poly,dtype=float).reshape(-1,2)
                    ax.plot(np.r_[pts[:,0],pts[0,0]],np.r_[pts[:,1],pts[0,1]],c=color,lw=.6)
        box=a.get('bbox')
        if isinstance(box,list) and len(box)==4:
            x,y,w,h=box
            ax.add_patch(Rectangle((x,y),w,h,fill=False,edgecolor=color,linewidth=.3))
    ax.set_title(f"{row['split']} / image={row['image_id']} / BB={row['BB']}, WB={row['WB']}")
    ax.axis('off')
for ax in axs[len(sampled[:6]):]: ax.axis('off')
fig.tight_layout()
fig.savefig(REPORT_DIR/'sample_mask_overlays.png',dpi=150,bbox_inches='tight')
plt.show()

---
## 7. Optional model experiment — **disabled by default**

The next sections document a future baseline and are safe to skip for EDA submission. Training does **not** run unless `PREPARE_YOLO=True` and `RUN_TRAIN=True` in the settings. The pipeline preserves the V2 supplied split, maps BB→0 and WB→1, and reports how many mask instances the conversion keeps or drops.

**Methodological note:** The original paper reports an AP protocol on a different release/split, so any V2 mAP@50:95 will be presented as a new experiment, not a direct replication. The conversion below retains only the largest polygon of multipart masks; this is a known approximation that must be quantified.

In [ ]:
YOLO_ROOT = WORK_ROOT/'yolo_v2'
RUNS_ROOT = DRIVE_ROOT/'model_runs'
BEST = None
conversion_stats = Counter()

if PREPARE_YOLO or RUN_TRAIN:
    # Optional packages for tomorrow's conversion; EDA does not depend on them.
    try:
        import yaml, cv2
        from pycocotools import mask as coco_mask
    except ImportError:
        subprocess.check_call([sys.executable,'-m','pip','-q','install',
                               'PyYAML','opencv-python-headless','pycocotools'])
        import yaml, cv2
        from pycocotools import mask as coco_mask
    from collections import OrderedDict
    from tqdm.auto import tqdm

    if YOLO_ROOT.exists():
        shutil.rmtree(YOLO_ROOT)
    for split in ('train','val','test'):
        (YOLO_ROOT/'images'/split).mkdir(parents=True,exist_ok=True)
        (YOLO_ROOT/'labels'/split).mkdir(parents=True,exist_ok=True)

    label_map = {}
    records = []
    for row in image_rows:
        split, iid = row['split'],int(row['image_id'])
        # Hash filenames, because filenames can be repeated between splits.
        key = f'{split}::{iid}::{row["coco_filename"]}'
        stem = hashlib.sha1(key.encode()).hexdigest()[:16]
        src = Path(row['image_path'])
        img_dst = YOLO_ROOT/'images'/split/(stem+src.suffix.lower())
        label_dst=YOLO_ROOT/'labels'/split/(stem+'.txt')
        if not img_dst.exists():
            try: img_dst.symlink_to(src)
            except OSError: shutil.copy2(src,img_dst)
        label_dst.write_text('',encoding='utf-8')
        label_map[(split,iid)] = (label_dst,row)
        records.append({'split':split,'image_id':iid,'source':str(src),'image':str(img_dst),'label':str(label_dst)})

    # Reuse a small pool of file handles for per-image label output.
    class LabelWriters:
        def __init__(self,max_open=24): self.max_open,self.open=max_open,OrderedDict()
        def write(self,path,line):
            key=str(path)
            if key in self.open:
                f=self.open.pop(key)
            else:
                if len(self.open)>=self.max_open:
                    _,old=self.open.popitem(last=False)
                    old.close()
                f=open(path,'a',encoding='utf-8')
            f.write(line+'\n')
            self.open[key]=f
        def close(self):
            for f in self.open.values(): f.close()
            self.open.clear()

    def polygon_area_2d(poly):
        xy=np.asarray(poly,dtype=np.float64).reshape(-1,2)
        return abs(float(np.dot(xy[:,0],np.roll(xy[:,1],-1)) -
                         np.dot(xy[:,1],np.roll(xy[:,0],-1))))/2

    def select_polygon(ann,H,W):
        seg=ann.get('segmentation')
        if isinstance(seg,list):
            choices=[]
            for p in seg:
                if not isinstance(p,list) or len(p)<6 or len(p)%2: continue
                try:
                    xy=np.asarray(p,dtype=np.float64).reshape(-1,2)
                    if np.isfinite(xy).all() and polygon_area_2d(p)>1: choices.append((polygon_area_2d(p),p))
                except (TypeError,ValueError): pass
            if len(choices)>1: conversion_stats['multipart_largest_only']+=1
            return max(choices,key=lambda x:x[0])[1] if choices else None
        if isinstance(seg,dict):
            try:
                rle=coco_mask.frPyObjects(seg,H,W) if isinstance(seg.get('counts'),list) else seg
                mask=coco_mask.decode(rle)
                if mask.ndim==3: mask=np.any(mask,axis=2).astype(np.uint8)
                contours,_=cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
                contours=[c for c in contours if len(c)>=3 and cv2.contourArea(c)>1]
                if len(contours)>1: conversion_stats['multipart_largest_only']+=1
                return max(contours,key=cv2.contourArea).reshape(-1).tolist() if contours else None
            except (ValueError,TypeError,KeyError): return None
        return None

    writer=LabelWriters()
    try:
        for split in ('train','val','test'):
            for kind,ann in coco_records(COCO_FILES[split]):
                if kind!='annotations': continue
                key=(split,int(ann['image_id']))
                if key not in label_map:
                    conversion_stats['unknown_image']+=1;continue
                label_dst,im=label_map[key]
                label=class_mapping.get((split,int(ann['category_id'])))
                if label not in ('BB','WB'):
                    conversion_stats['unmapped_class']+=1;continue
                if ann.get('iscrowd',0):
                    conversion_stats['crowd_excluded']+=1;continue
                p=select_polygon(ann, int(im['height']),int(im['width']))
                if p is None:
                    conversion_stats['invalid_or_missing_polygon']+=1;continue
                coords=np.asarray(p,dtype=np.float64).reshape(-1,2)
                if not np.isfinite(coords).all():
                    conversion_stats['nonfinite_polygon']+=1;continue
                coords[:,0] = np.clip(coords[:,0]/im['width'],0,1)
                coords[:,1] = np.clip(coords[:,1]/im['height'],0,1)
                klass=0 if label=='BB' else 1
                writer.write(label_dst, str(klass)+' '+' '.join(f'{v:.6f}' for v in coords.ravel()))
                conversion_stats['written_instances']+=1
    finally:
        writer.close()

    data_yaml={'path':str(YOLO_ROOT.resolve()),'train':'images/train',
               'val':'images/val','test':'images/test','names':{0:'BB',1:'WB'}}
    with (YOLO_ROOT/'data.yaml').open('w') as f:
        yaml.safe_dump(data_yaml,f,sort_keys=False)
    pd.DataFrame(records).to_csv(REPORT_DIR/'yolo_conversion_manifest.csv',index=False)
    (REPORT_DIR/'yolo_conversion_stats.json').write_text(json.dumps(dict(conversion_stats),indent=2))
    print('YOLO data:',YOLO_ROOT/'data.yaml')
    print('Conversion:',dict(conversion_stats))
else:
    print('YOLO conversion skipped; EDA is complete without a model dataset export.')

## 8. Optional training and held-out evaluation

The run uses a 640-pixel YOLOv8n-seg pilot initialized from pretrained weights. Training checkpoints are saved on Drive. Test evaluation is separate and **opt-in**, so exploratory choices cannot accidentally tune against the test split. GPU availability and I/O will determine runtime.

In [ ]:
from google.colab import drive
from pathlib import Path
from collections import Counter
import csv
import json
import shutil
import zipfile
import os

drive.mount('/content/drive')

# Đường dẫn Google Drive
DRIVE_ROOT = Path('/content/drive/MyDrive/M18K_Reproduction (1)')
DATA_ROOT = DRIVE_ROOT / 'M18K_V2_data' / 'M18KV2'

# File ZIP chứa nhãn WB
ZIP_PATH = DRIVE_ROOT / 'derived_datasets' / 'wb_yolo_labels.zip'

# Dataset WB-only sau khi chuẩn bị
WORK_ROOT = Path('/content/M18K_V2_work')
WB_ROOT = WORK_ROOT / 'yolo_v2_wb'

# Thư mục lưu kết quả training
RUNS_ROOT = DRIVE_ROOT / 'model_runs'

# Cấu hình training
SEED = 42
MODEL_NAME = 'yolov8n-seg.pt'
IMAGE_SIZE = 640
EPOCHS = 10
BATCH = 8

RUN_TRAIN = True
RUN_FULL_CONTENT_HASH_AUDIT = False

WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.is_file(), f'Không tìm thấy ZIP: {ZIP_PATH}'
assert DATA_ROOT.is_dir(), f'Không tìm thấy thư mục ảnh: {DATA_ROOT}'

print('Drive mounted successfully')
print('ZIP:', ZIP_PATH)
print('Original images:', DATA_ROOT)
print('WB dataset:', WB_ROOT)

In [ ]:
EXPECTED = {
    'train': 3596,
    'val': 449,
    'test': 450
}

# Xóa dataset cũ nếu đã tồn tại
if WB_ROOT.exists():
    shutil.rmtree(WB_ROOT)

for split in ['train', 'val', 'test']:
    (WB_ROOT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (WB_ROOT / 'labels' / split).mkdir(parents=True, exist_ok=True)

stats = Counter()

with zipfile.ZipFile(ZIP_PATH, 'r') as z:

    members = set(z.namelist())

    # Kiểm tra cấu trúc ZIP
    assert 'image_map.csv' in members, 'ZIP thiếu image_map.csv'

    for split in ['train', 'val', 'test']:
        assert f'labels/{split}/' in members or any(
            name.startswith(f'labels/{split}/') for name in members
        ), f'ZIP thiếu labels/{split}/'

    # Giải nén nhãn
    for name in members:
        if name.startswith('labels/') and name.endswith('.txt'):
            relative_path = Path(name).relative_to('labels')
            destination = WB_ROOT / 'labels' / relative_path
            destination.parent.mkdir(parents=True, exist_ok=True)

            destination.write_bytes(z.read(name))

    # Đọc file ánh xạ ảnh
    with z.open('image_map.csv') as f:
        rows = list(
            csv.DictReader(
                line.decode('utf-8-sig')
                for line in f
            )
        )

    assert len(rows) == sum(EXPECTED.values()), (
        f'Số dòng image_map.csv không đúng: {len(rows)}'
    )

    for row in rows:
        split = row['split']
        original_rgb = row['original_rgb']
        image_name = row['image_name']

        if split not in EXPECTED:
            raise ValueError(f'Split không hợp lệ: {split}')

        # Đường dẫn ảnh gốc
        src_image = DATA_ROOT / original_rgb

        if not src_image.is_file():
            raise FileNotFoundError(
                f'Không tìm thấy ảnh gốc: {src_image}'
            )

        # Đường dẫn ảnh và nhãn đích
        dst_image = WB_ROOT / 'images' / split / image_name

        dst_label = (
            WB_ROOT / 'labels' / split /
            (Path(image_name).stem + '.txt')
        )

        if not dst_label.is_file():
            raise FileNotFoundError(
                f'Không tìm thấy nhãn: {dst_label}'
            )

        # Kiểm tra class: chỉ được có class 0 = WB
        label_text = dst_label.read_text(encoding='utf-8')

        for line in label_text.splitlines():
            parts = line.split()

            if parts and parts[0] != '0':
                raise ValueError(
                    f'Phát hiện class khác WB trong {dst_label}'
                )

        # Tạo symlink, không copy ảnh
        dst_image.symlink_to(src_image.resolve())

        stats[split] += 1

# Kiểm tra số lượng
assert dict(stats) == EXPECTED, (
    f'Số lượng ảnh không đúng: {dict(stats)}'
)

# Tạo data.yaml
data_yaml = f"""path: {WB_ROOT}
train: images/train
val: images/val
test: images/test

nc: 1
names:
  0: WB
"""

(WB_ROOT / 'data.yaml').write_text(
    data_yaml,
    encoding='utf-8'
)

print('WB-only dataset created successfully!')
print('Image counts:', dict(stats))
print('Data YAML:', WB_ROOT / 'data.yaml')

In [ ]:
import yaml

DATA_YAML = WB_ROOT / 'data.yaml'

with open(DATA_YAML, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

assert config['nc'] == 1
assert config['names'][0] == 'WB'

expected = {
    'train': 3596,
    'val': 449,
    'test': 450
}

for split, expected_count in expected.items():

    image_dir = WB_ROOT / 'images' / split
    label_dir = WB_ROOT / 'labels' / split

    images = list(image_dir.iterdir())
    labels = list(label_dir.glob('*.txt'))

    assert len(images) == expected_count, (
        f'{split}: sai số lượng ảnh'
    )

    assert len(labels) == expected_count, (
        f'{split}: sai số lượng nhãn'
    )

    # Kiểm tra ảnh liên kết còn tồn tại
    for image_path in images:
        assert image_path.is_file(), (
            f'Ảnh không truy cập được: {image_path}'
        )

    # Kiểm tra class
    for label_file in labels:
        for line in label_file.read_text(
            encoding='utf-8'
        ).splitlines():

            parts = line.split()

            if parts:
                assert parts[0] == '0', (
                    f'Phát hiện class khác WB: {label_file}'
                )

    print(
        f'{split}: {len(images)} images, '
        f'{len(labels)} labels - OK'
    )

print('\nDATASET VALIDATION PASSED')
print('Number of classes:', config['nc'])
print('Class names:', config['names'])

In [ ]:

if RUN_TRAIN:
    DATA_YAML = WB_ROOT / 'data.yaml'

    if not DATA_YAML.is_file():
        raise RuntimeError(
            f'WB dataset YAML not found: {DATA_YAML}. '
            'Run the 3 preparation cells first.'
        )

    import subprocess
    subprocess.check_call([
        sys.executable, '-m', 'pip', '-q',
        'install', 'ultralytics>=8.3,<9'
    ])

    import torch
    from ultralytics import YOLO

    if not torch.cuda.is_available():
        raise RuntimeError(
            'GPU unavailable. Choose a GPU runtime for training.'
        )

    RUNS_ROOT.mkdir(parents=True, exist_ok=True)

    run_name = (
        f'wb_only_yolov8n_seg_{IMAGE_SIZE}_'
        f'e{EPOCHS}_s{SEED}'
    )

    model = YOLO(MODEL_NAME)

    model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH,
        optimizer='AdamW',
        lr0=1e-4,
        cos_lr=True,
        degrees=15,
        translate=.1,
        scale=.2,
        mosaic=0,
        fliplr=0,
        flipud=0,
        mixup=0,
        seed=SEED,
        project=str(RUNS_ROOT),
        name=run_name,
        exist_ok=False,
        device=0,
        workers=2,
        plots=True,
        save_period=1
    )

    BEST = RUNS_ROOT / run_name / 'weights' / 'best.pt'

    print(
        'Best checkpoint:', BEST,
        '| exists:', BEST.is_file()
    )

else:
    print('Training disabled. No model accuracy claimed.')


In [ ]:
if RUN_TEST_EVAL:
    import subprocess
    subprocess.check_call([sys.executable,'-m','pip','-q','install','ultralytics>=8.3,<9'])
    import torch
    from ultralytics import YOLO
    if not (YOLO_ROOT/'data.yaml').is_file():
        raise RuntimeError('Test dataset YAML not found: prepare YOLO dataset first.')
    if BEST is None:
        candidate=RUNS_ROOT/f'v2_yolov8n_seg_{IMAGE_SIZE}_e{EPOCHS}_s{SEED}'/'weights'/'best.pt'
        if candidate.exists(): BEST=candidate
    if BEST is None or not BEST.is_file():
        raise FileNotFoundError('No trained best.pt found. Cannot evaluate test split.')
    result=YOLO(str(BEST)).val(data=str(YOLO_ROOT/'data.yaml'),split='test',
                               imgsz=IMAGE_SIZE,batch=BATCH,
                               device=0 if torch.cuda.is_available() else 'cpu',
                               plots=True,project=str(RUNS_ROOT),name='heldout_test')
    metrics={
        'checkpoint':str(BEST),'dataset':'M18K V2 (not identical to paper split)',
        'mask_mAP50':float(result.seg.map50),'mask_mAP50_95':float(result.seg.map),
        'box_mAP50':float(result.box.map50),'box_mAP50_95':float(result.box.map),
        'test_split':'dataset-supplied test',
        'directly_comparable_to_paper':False
    }
    (REPORT_DIR/'test_metrics.json').write_text(json.dumps(metrics,indent=2))
    display(pd.DataFrame([metrics]).T)
else:
    print('Test evaluation disabled; no held-out model result is claimed.')

### Interpretation and next analysis

The immediate deliverable is a **reproducible V2 dataset audit**, not a claim that the original paper has been reproduced. The next experiment should verify the unusually rare BB labels by visual inspection, document any missing depth matches, check scene-level separation, and measure test-set segmentation AP only after training is complete. Larger images, RGB–depth fusion and 3D pose estimation belong to separate controlled experiments.

In [ ]:
from pathlib import Path
from collections import Counter
import csv, io, json, zipfile

YOLO_ROOT = Path("/content/M18K_V2_work/yolo_v2")
DRIVE_ROOT = Path("/content/drive/MyDrive/M18K_Reproduction")
MANIFEST = DRIVE_ROOT / "reports/v2_eda/yolo_conversion_manifest.csv"

OUT = DRIVE_ROOT / "derived_datasets/wb_yolo_labels.zip"

assert (YOLO_ROOT / "labels/train").is_dir(), (
    "Original YOLO labels not found. Colab runtime may have reset."
)
assert MANIFEST.is_file(), "Missing original conversion manifest."

OUT.parent.mkdir(parents=True, exist_ok=True)

counts = Counter()
mapping = []
seen = set()

with MANIFEST.open(newline="") as f, zipfile.ZipFile(
    OUT, "w", zipfile.ZIP_DEFLATED, compresslevel=3
) as z:

    for row in csv.DictReader(f):
        split = row["split"]  # train / val / test
        original = Path(row["source"])
        original_rel = row["source"].split("/M18KV2/", 1)[-1]

        # Restore original image filenames, instead of hashed YOLO names
        filename = original.stem + ".txt"
        key = (split, filename)
        assert key not in seen, f"Duplicate image stem: {key}"
        seen.add(key)

        old_label = (
            YOLO_ROOT / "labels" / split / Path(row["label"]).name
        )
        assert old_label.is_file(), f"Missing label: {old_label}"

        wb_lines = []

        for line in old_label.read_text().splitlines():
            parts = line.split()
            if not parts:
                continue

            if parts[0] == "1":  # WB -> class 0
                wb_lines.append("0 " + " ".join(parts[1:]))
                counts["WB_instances"] += 1
            elif parts[0] == "0":  # Discard BB
                counts["BB_excluded"] += 1
            else:
                raise ValueError(f"Unexpected class: {parts[0]}")

        z.writestr(
            f"labels/{split}/{filename}",
            "\n".join(wb_lines) + ("\n" if wb_lines else "")
        )

        counts[f"{split}_images"] += 1
        mapping.append({
            "split": split,
            "original_rgb": original_rel,
            "image_name": original.name,
        })

    buffer = io.StringIO()
    writer = csv.DictWriter(
        buffer, fieldnames=["split", "original_rgb", "image_name"]
    )
    writer.writeheader()
    writer.writerows(mapping)
    z.writestr("image_map.csv", buffer.getvalue())

    z.writestr("dataset_info.json", json.dumps({
        "classes": {"0": "WB"},
        "source": "M18K V2, original two-class YOLO labels",
        "conversion": "BB excluded; WB class 1 remapped to 0",
        "counts": dict(counts)
    }, indent=2))

assert [counts[f"{s}_images"] for s in ("train", "val", "test")] == [
    3596, 449, 450
]

print("SUCCESS:", OUT)
print("Stats:", dict(counts))
print("Size (MB):", round(OUT.stat().st_size / 1e6, 2))

INFERENCE


In [ ]:
!pip -q install "ultralytics>=8.3,<9" pycocotools pyyaml

from google.colab import drive
drive.mount('/content/drive')

import os
import re
import json
import csv
import cv2
import yaml
import hashlib
import ultralytics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from collections import defaultdict
from pycocotools.coco import COCO
from ultralytics import YOLO

# =====================================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# =====================================

BASE_DIR = Path('/content/drive/MyDrive/M18K_Reproduction')

CHECKPOINT_CANDIDATES = [
    BASE_DIR / 'best.pt',
    BASE_DIR / 'model_runs/khang_wb_heldout_test/weights/best.pt'
]

CHECKPOINT = next(
    (p for p in CHECKPOINT_CANDIDATES if p.is_file()), None
)

if CHECKPOINT is None:
    raise FileNotFoundError(
        'Không tìm thấy best.pt. Các đường dẫn đã kiểm tra:\n' +
        '\n'.join(map(str, CHECKPOINT_CANDIDATES))
    )

DATA_YAML = BASE_DIR / 'data.yaml'

if not DATA_YAML.is_file():
    raise FileNotFoundError(f'Không tìm thấy: {DATA_YAML}')

TEST_DIR_CANDIDATES = [
    BASE_DIR / 'M18K_V2_data/M18KV2/test',
    BASE_DIR / 'M18K_V2_data/test'
]

TEST_DIR = next(
    (p for p in TEST_DIR_CANDIDATES if (p / 'rgb').is_dir()),
    None
)

if TEST_DIR is None:
    raise FileNotFoundError('Không tìm thấy thư mục test/rgb.')

RGB_DIR = TEST_DIR / 'rgb'

ANN_CANDIDATES = [
    TEST_DIR / 'annotations_coco.json',
    BASE_DIR / 'M18K_V2_data/annotations_coco.json'
]

ANN_PATH = next(
    (p for p in ANN_CANDIDATES if p.is_file()),
    None
)

if ANN_PATH is None:
    matches = list(
        (BASE_DIR / 'M18K_V2_data').rglob('annotations_coco.json')
    )
    if len(matches) == 1:
        ANN_PATH = matches[0]
    else:
        raise FileNotFoundError(
            f'Không xác định được annotations_coco.json: {matches}'
        )

RESULTS_DIR = BASE_DIR / '09_results/wb_error_analysis'
PRED_DIR = RESULTS_DIR / 'prediction_examples'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR.mkdir(parents=True, exist_ok=True)

# =====================================
# 2. THAM SỐ INFERENCE
# =====================================

IMAGE_SIZE = 640
CONF_THRES = 0.25
NMS_IOU = 0.70
MAX_DET = 3000

# Ngưỡng ghép mask để phân tích mẫu.
MATCH_IOU = 0.50
BOUNDARY_IOU = 0.75

RUN_SIMILARITY_SPOTCHECK = True

# Metrics đã hoàn thành trước đó.
# Không chạy lại đánh giá toàn bộ 450 ảnh.
KNOWN_METRICS = {
    'test_images': 450,
    'wb_instances': 70192,
    'mask_mAP50': 0.990639,
    'mask_mAP50_95': 0.797363,
    'mask_recall': 0.963078
}

print('Checkpoint:', CHECKPOINT)
print('Data YAML:', DATA_YAML)
print('Test RGB:', RGB_DIR)
print('COCO annotation:', ANN_PATH)
print('Results:', RESULTS_DIR)

# Xác nhận WB-only mapping trong data.yaml.
with open(DATA_YAML, 'r', encoding='utf-8') as f:
    yaml_cfg = yaml.safe_load(f)

names = yaml_cfg.get('names', {})

if isinstance(names, list):
    class_zero = names[0]
else:
    class_zero = names.get(0, names.get('0'))

assert yaml_cfg.get('nc') == 1, 'Dataset không có nc=1.'
assert str(class_zero).upper() == 'WB', 'Class 0 không phải WB.'

print('Đã xác nhận: nc=1, class 0=WB')

In [ ]:
coco = COCO(str(ANN_PATH))

# =====================================
# 1. TÌM CATEGORY WB
# =====================================

categories = coco.dataset.get('categories', [])

def normalize_name(name):
    return re.sub(r'[^a-z0-9]+', ' ', str(name).lower()).strip()

wb_categories = []

for cat in categories:
    name = normalize_name(cat.get('name', ''))

    if (
        name == 'wb'
        or ('white' in name and 'button' in name)
    ):
        wb_categories.append(cat)

if len(wb_categories) == 1:
    WB_CATEGORY = wb_categories[0]
elif len(categories) == 1:
    WB_CATEGORY = categories[0]
else:
    raise ValueError(
        f'Không xác định được category WB. Categories: {categories}'
    )

WB_CAT_ID = int(WB_CATEGORY['id'])

print('WB category:', WB_CATEGORY)

# =====================================
# 2. MAP COCO IMAGES VỚI FILE RGB
# =====================================

image_files = [
    p for p in RGB_DIR.rglob('*')
    if p.is_file() and p.suffix.lower() in
    {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
]

by_name = defaultdict(list)

for p in image_files:
    by_name[p.name].append(p)

anns_by_image = defaultdict(list)

for ann in coco.dataset['annotations']:
    if int(ann['category_id']) == WB_CAT_ID:
        anns_by_image[int(ann['image_id'])].append(ann)

records = []
missing_images = []

for im in coco.dataset['images']:

    image_id = int(im['id'])
    filename = Path(im['file_name']).name

    candidates = by_name.get(filename, [])

    if not candidates:
        missing_images.append(filename)
        continue

    if len(candidates) > 1:
        raise RuntimeError(f'Tên ảnh bị trùng: {filename}')

    boxes = [
        ann.get('bbox', [0, 0, 0, 0])
        for ann in anns_by_image[image_id]
    ]

    width = float(im.get('width', 1))
    height = float(im.get('height', 1))

    areas = [
        max(0, float(b[2])) * max(0, float(b[3]))
        / max(1, width * height)
        for b in boxes
    ]

    # Ước lượng độ chồng lấn giữa các bounding box.
    overlap_values = []

    for i, a in enumerate(boxes):

        ax, ay, aw, ah = map(float, a[:4])
        best_overlap = 0.0

        for j, b in enumerate(boxes):

            if i == j:
                continue

            bx, by, bw, bh = map(float, b[:4])

            ix1 = max(ax, bx)
            iy1 = max(ay, by)
            ix2 = min(ax + aw, bx + bw)
            iy2 = min(ay + ah, by + bh)

            inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)

            union = (
                aw * ah + bw * bh - inter
            )

            iou = inter / union if union > 0 else 0

            best_overlap = max(best_overlap, iou)

        overlap_values.append(best_overlap)

    records.append({
        'image_id': image_id,
        'filename': filename,
        'image_path': candidates[0],
        'gt_count': len(boxes),
        'median_bbox_area_fraction': (
            float(np.median(areas)) if areas else 0
        ),
        'bbox_overlap_score': (
            float(np.mean(overlap_values))
            if overlap_values else 0
        )
    })

if len(records) < 12:
    raise RuntimeError(
        f'Chỉ tìm thấy {len(records)} ảnh hợp lệ, cần ít nhất 12.'
    )

print('Số ảnh map được:', len(records))
print('Số ảnh không tìm thấy:', len(missing_images))

# =====================================
# 3. CHỌN 4 DENSE
# =====================================

dense = sorted(
    records,
    key=lambda x: (-x['gt_count'], x['image_id'])
)[:4]

used = {x['image_id'] for x in dense}

# =====================================
# 4. CHỌN 4 SPARSE
# =====================================

remaining = [
    x for x in records
    if x['image_id'] not in used and x['gt_count'] > 0
]

sparse = sorted(
    remaining,
    key=lambda x: (x['gt_count'], x['image_id'])
)[:4]

used.update(x['image_id'] for x in sparse)

# =====================================
# 5. CHỌN 4 DIFFICULT CANDIDATES
# =====================================

remaining = [
    x for x in records
    if x['image_id'] not in used
]

small_values = [
    x['median_bbox_area_fraction']
    for x in remaining
]

vmin = min(small_values)
vmax = max(small_values)

for x in remaining:

    if vmax > vmin:
        smallness = 1 - (
            (x['median_bbox_area_fraction'] - vmin)
            / (vmax - vmin)
        )
    else:
        smallness = 0.5

    x['difficulty_score'] = (
        0.5 * smallness
        + 0.5 * x['bbox_overlap_score']
    )

difficult = sorted(
    remaining,
    key=lambda x: (
        -x['difficulty_score'],
        -x['gt_count'],
        x['image_id']
    )
)[:4]

# =====================================
# 6. TẠO SELECTED EXAMPLES
# =====================================

selected = []

for group, items in [
    ('dense', dense),
    ('sparse', sparse),
    ('difficult', difficult)
]:

    for x in items:

        if group == 'dense':
            reason = (
                f"Top mật độ theo GT: "
                f"{x['gt_count']} WB instances."
            )

        elif group == 'sparse':
            reason = (
                f"Mật độ thấp theo GT: "
                f"{x['gt_count']} WB instances."
            )

        else:
            reason = (
                'Ứng viên khó dựa trên bbox area và bbox overlap. '
                'Cần xác nhận bằng mắt; chưa khẳng định có occlusion.'
            )

        selected.append({
            **x,
            'selection_group': group,
            'reason': reason
        })

selected_path = RESULTS_DIR / 'selected_examples.csv'

with open(selected_path, 'w', newline='', encoding='utf-8-sig') as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            'image_id',
            'filename',
            'selection_group',
            'reason'
        ]
    )

    writer.writeheader()

    for x in selected:
        writer.writerow({
            k: x[k]
            for k in [
                'image_id',
                'filename',
                'selection_group',
                'reason'
            ]
        })

display(pd.DataFrame([
    {
        'image_id': x['image_id'],
        'filename': x['filename'],
        'group': x['selection_group'],
        'GT instances': x['gt_count'],
        'reason': x['reason']
    }
    for x in selected
]))

print('Đã tạo selected_examples.csv')

In [ ]:
model = YOLO(str(CHECKPOINT))

def get_gt_masks(image_id, shape):

    masks = []

    for ann in anns_by_image[int(image_id)]:

        mask = coco.annToMask(ann).astype(np.uint8)

        if mask.shape != shape:
            mask = cv2.resize(
                mask,
                (shape[1], shape[0]),
                interpolation=cv2.INTER_NEAREST
            )

        if mask.any():
            masks.append(mask.astype(bool))

    return masks


def get_pred_masks(result, shape):

    if result.masks is None:
        return []

    raw = result.masks.data.cpu().numpy()
    masks = []

    for mask in raw:

        mask = (mask > 0.5).astype(np.uint8)

        if mask.shape != shape:
            mask = cv2.resize(
                mask,
                (shape[1], shape[0]),
                interpolation=cv2.INTER_NEAREST
            )

        if mask.any():
            masks.append(mask.astype(bool))

    return masks


def calculate_iou(a, b):

    intersection = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()

    return intersection / union if union else 0


def match_masks(gt_masks, pred_masks, threshold=0.50):

    if not gt_masks or not pred_masks:
        return [], len(gt_masks), len(pred_masks)

    matrix = np.zeros(
        (len(gt_masks), len(pred_masks)),
        dtype=np.float32
    )

    for i, gt in enumerate(gt_masks):
        for j, pred in enumerate(pred_masks):
            matrix[i, j] = calculate_iou(gt, pred)

    pairs = []
    used_gt = set()
    used_pred = set()

    # Ghép cặp một-một theo IoU cao nhất.
    while True:

        work = matrix.copy()

        if used_gt:
            work[list(used_gt), :] = -1

        if used_pred:
            work[:, list(used_pred)] = -1

        i, j = np.unravel_index(
            np.argmax(work), work.shape
        )

        value = float(work[i, j])

        if value < threshold:
            break

        pairs.append((i, j, value))
        used_gt.add(i)
        used_pred.add(j)

    missed = len(gt_masks) - len(used_gt)
    extra = len(pred_masks) - len(used_pred)

    return pairs, missed, extra


def overlay_masks(rgb, masks, color):

    out = rgb.copy()

    for mask in masks:

        color_array = np.array(color, dtype=np.float32)

        out[mask] = (
            0.55 * out[mask].astype(np.float32)
            + 0.45 * color_array
        ).astype(np.uint8)

        contours, _ = cv2.findContours(
            mask.astype(np.uint8),
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        cv2.drawContours(out, contours, -1, color, 1)

    return out


observations = []

for index, item in enumerate(selected, start=1):

    image_path = item['image_path']

    bgr = cv2.imread(str(image_path))

    if bgr is None:
        raise RuntimeError(f'Không đọc được ảnh: {image_path}')

    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    h, w = rgb.shape[:2]

    gt_masks = get_gt_masks(
        item['image_id'],
        (h, w)
    )

    # Chỉ inference trên từng ảnh đã chọn.
    result = model.predict(
        source=str(image_path),
        imgsz=IMAGE_SIZE,
        conf=CONF_THRES,
        iou=NMS_IOU,
        max_det=MAX_DET,
        classes=[0],
        retina_masks=True,
        agnostic_nms=False,
        save=False,
        verbose=False
    )[0]

    pred_masks = get_pred_masks(
        result,
        (h, w)
    )

    pairs, missed, extra = match_masks(
        gt_masks,
        pred_masks,
        MATCH_IOU
    )

    matched_ious = [x[2] for x in pairs]

    mean_iou = (
        float(np.mean(matched_ious))
        if matched_ious else None
    )

    boundary_candidates = sum(
        x < BOUNDARY_IOU for x in matched_ious
    )

    issue_parts = []

    if missed:
        issue_parts.append('missed')

    if extra:
        issue_parts.append('extra')

    if not missed and not extra and boundary_candidates:
        issue_parts.append('boundary_candidate')

    issue_type = (
        '+'.join(issue_parts)
        if issue_parts else 'none'
    )

    notes = [
        f'GT={len(gt_masks)}',
        f'Prediction={len(pred_masks)}',
        f'Matched@IoU50={len(pairs)}'
    ]

    if missed:
        notes.append(
            f'{missed} GT masks chưa được ghép'
        )

    if extra:
        notes.append(
            f'{extra} prediction masks chưa được ghép'
        )

    if boundary_candidates:
        notes.append(
            f'{boundary_candidates} cặp có IoU dưới {BOUNDARY_IOU}'
        )

    notes.append('Chờ human review')

    # Tạo hình Original / GT / Prediction.
    gt_overlay = overlay_masks(
        rgb, gt_masks, (40, 190, 80)
    )

    pred_overlay = overlay_masks(
        rgb, pred_masks, (240, 90, 50)
    )

    fig, axes = plt.subplots(
        1, 3,
        figsize=(18, 6)
    )

    axes[0].imshow(rgb)
    axes[0].set_title(
        f"Original - ID {item['image_id']}"
    )

    axes[1].imshow(gt_overlay)
    axes[1].set_title(
        f"Ground Truth - {len(gt_masks)} WB"
    )

    axes[2].imshow(pred_overlay)
    axes[2].set_title(
        f"Prediction - {len(pred_masks)} WB"
    )

    for ax in axes:
        ax.axis('off')

    fig.suptitle(
        f"{item['filename']} | {issue_type}"
    )

    fig.tight_layout()

    evidence_name = (
        f"img_{item['image_id']}_{image_path.stem}.png"
    )

    fig.savefig(
        PRED_DIR / evidence_name,
        dpi=160,
        bbox_inches='tight'
    )

    plt.close(fig)

    observations.append({
        'image_id': item['image_id'],
        'filename': item['filename'],
        'issue_type': issue_type,
        'observation': '; '.join(notes),
        'evidence_filename': evidence_name,
        'gt_instances': len(gt_masks),
        'pred_instances': len(pred_masks),
        'matched_instances_iou50': len(pairs),
        'missed_instances_iou50': missed,
        'extra_instances_iou50': extra,
        'mean_matched_mask_iou': (
            round(mean_iou, 4)
            if mean_iou is not None else ''
        ),
        'manual_review_status': 'PENDING',
        'manual_review_notes': ''
    })

    print(
        f"[{index}/12] {item['filename']} | "
        f"{issue_type} | GT={len(gt_masks)} | "
        f"Pred={len(pred_masks)} | "
        f"Missed={missed} | Extra={extra}"
    )

# Xuất CSV observations.
obs_path = RESULTS_DIR / 'error_observations.csv'

with open(obs_path, 'w', newline='', encoding='utf-8-sig') as f:

    writer = csv.DictWriter(
        f,
        fieldnames=list(observations[0].keys())
    )

    writer.writeheader()
    writer.writerows(observations)

print('Đã xuất 12 ảnh comparison.')
print('Đã tạo error_observations.csv')

In [ ]:
def sha256_file(path):

    sha = hashlib.sha256()

    with open(path, 'rb') as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b''
        ):
            sha.update(chunk)

    return sha.hexdigest()


config = {
    'checkpoint_filename': CHECKPOINT.name,
    'checkpoint_path': str(CHECKPOINT),
    'checkpoint_sha256': sha256_file(CHECKPOINT),
    'data_yaml_path': str(DATA_YAML),
    'test_rgb_dir': str(RGB_DIR),
    'test_annotation_path': str(ANN_PATH),
    'model_classes': {'0': 'WB'},
    'coco_wb_category': WB_CATEGORY,
    'imgsz': IMAGE_SIZE,
    'confidence_threshold': CONF_THRES,
    'nms_iou_threshold': NMS_IOU,
    'max_det': MAX_DET,
    'classes_filter': [0],
    'retina_masks': True,
    'agnostic_nms': False,
    'ultralytics_version': ultralytics.__version__,
    'matching_mask_iou_threshold': MATCH_IOU,
    'boundary_review_threshold': BOUNDARY_IOU,
    'existing_test_metrics': KNOWN_METRICS
}

with open(
    RESULTS_DIR / 'inference_config.json',
    'w',
    encoding='utf-8'
) as f:
    json.dump(config, f, indent=2, ensure_ascii=False)


# =====================================
# SIMILARITY SPOT-CHECK
# =====================================

def dhash64(path):

    image = cv2.imread(
        str(path),
        cv2.IMREAD_GRAYSCALE
    )

    if image is None:
        return None

    small = cv2.resize(
        image,
        (9, 8),
        interpolation=cv2.INTER_AREA
    )

    bits = (small[:, 1:] > small[:, :-1]).flatten()

    value = 0

    for bit in bits:
        value = (value << 1) | int(bit)

    return value


TRAIN_RGB_DIR = TEST_DIR.parent / 'train/rgb'

spotcheck_status = 'NOT_VERIFIED'
spotcheck_rows = []

if RUN_SIMILARITY_SPOTCHECK and TRAIN_RGB_DIR.is_dir():

    train_files = [
        p for p in TRAIN_RGB_DIR.rglob('*')
        if p.is_file() and p.suffix.lower() in
        {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
    ]

    train_hashes = [
        (p, dhash64(p))
        for p in train_files
    ]

    train_hashes = [
        (p, h) for p, h in train_hashes
        if h is not None
    ]

    for item in selected:

        test_hash = dhash64(item['image_path'])

        if test_hash is None or not train_hashes:
            continue

        nearest_path, nearest_hash = min(
            train_hashes,
            key=lambda x: (test_hash ^ x[1]).bit_count()
        )

        distance = (
            test_hash ^ nearest_hash
        ).bit_count()

        if distance <= 3:
            label = 'high_similarity_candidate'
        elif distance <= 8:
            label = 'moderate_similarity_candidate'
        else:
            label = 'no_close_match_by_dhash'

        spotcheck_rows.append({
            'test_image_id': item['image_id'],
            'test_filename': item['filename'],
            'nearest_train_filename': nearest_path.name,
            'dhash_hamming_distance': distance,
            'screening_label': label
        })

    if spotcheck_rows:

        spotcheck_status = (
            'SPOT_CHECKED_DHASH_NOT_FULL_LEAKAGE_CLEARANCE'
        )

        pd.DataFrame(spotcheck_rows).to_csv(
            RESULTS_DIR / 'similarity_spotcheck.csv',
            index=False,
            encoding='utf-8-sig'
        )

elif RUN_SIMILARITY_SPOTCHECK:

    spotcheck_status = 'NOT_VERIFIED_TRAIN_RGB_NOT_FOUND'


# =====================================
# FINDINGS REPORT
# =====================================

missed_total = sum(
    int(x['missed_instances_iou50'])
    for x in observations
)

extra_total = sum(
    int(x['extra_instances_iou50'])
    for x in observations
)

boundary_total = sum(
    1 for x in observations
    if 'boundary_candidate' in x['issue_type']
)

clean_total = sum(
    1 for x in observations
    if x['issue_type'] == 'none'
)

findings = f"""
# WB-Only YOLOv8n-seg Error Analysis Findings

## 1. Scope

- Checkpoint: `{CHECKPOINT.name}`
- Dataset: M18K V2 supplied test split.
- Inference performed only on 12 selected images.
- No retraining was performed.
- Full 450-image test evaluation was not rerun.
- WB class mapping: class ID 0 = WB.

## 2. Existing Test Metrics

- Test images: {KNOWN_METRICS['test_images']}
- WB instances: {KNOWN_METRICS['wb_instances']:,}
- Mask AP@50: {KNOWN_METRICS['mask_mAP50']:.6f}
- Mask AP@50:95: {KNOWN_METRICS['mask_mAP50_95']:.6f}
- Mask Recall: {KNOWN_METRICS['mask_recall']:.6f}

These values are existing metrics supplied for this analysis.
They were not recalculated in this notebook.

## 3. Selected Examples

- 4 dense images: selected by highest GT instance counts.
- 4 sparse images: selected by lowest non-zero GT instance counts.
- 4 difficult candidates: selected using bbox area and bbox overlap heuristics.

The difficult candidates require visual confirmation.
BBox overlap alone does not prove physical occlusion.

## 4. Automated Sample Analysis

Mask matching threshold: IoU >= {MATCH_IOU:.2f}.

- Unmatched GT masks (possible misses): {missed_total}
- Unmatched prediction masks (possible false positives or duplicates): {extra_total}
- Images with boundary-review candidates: {boundary_total}
- Images without count mismatch at the matching threshold: {clean_total}

These are sample-level diagnostics for 12 images.
They are not the official mAP calculation.

All rows in `error_observations.csv` remain marked PENDING
until a person visually reviews the corresponding PNG.

## 5. Similarity Spot-check

Status: `{spotcheck_status}`.

The dHash check is a preliminary near-duplicate screen
for selected test images against available train images.
It is not a full train/test leakage audit and cannot prove
that the complete dataset is leakage-free.

## 6. Interpretation

The existing held-out metrics indicate strong aggregate WB
segmentation performance, particularly at IoU 0.50.

The lower AP@50:95 compared with AP@50 makes stricter mask
overlap and boundary alignment useful areas for qualitative
inspection.

The 12 selected images are a small heuristic-based sample.
They should not be treated as a statistically representative
estimate of all test scenes.

No two-class baseline comparison is included.

## 7. Completion Status

PARTIAL until the 12 comparison figures have been visually
reviewed and the manual-review fields in error_observations.csv
have been completed.
"""

with open(
    RESULTS_DIR / 'wb_error_findings.md',
    'w',
    encoding='utf-8'
) as f:
    f.write(findings)

print('✅ Đã hoàn thành tạo báo cáo.')

print('\nCác file trong thư mục kết quả:')

for p in sorted(RESULTS_DIR.iterdir()):
    print(p.name)